# Maji Ndogo Weather Data Validation

## Project Overview

This notebook focuses on validating weather station data collected from farms in Maji Ndogo. The objective is to prepare the weather data, calculate average measurements for each weather station, and compare these values with the main agricultural dataset to identify inconsistencies and assess data quality.

## Dataset

The analysis uses data from:

- Weather station measurements, including rainfall, temperature, and pollution level.
- The main agricultural dataset containing farm and environmental information.
- A weather station mapping table used to link each farm field to its corresponding weather station.

## Importing Libraries and Connecting to the Database

The required Python libraries were imported for data manipulation, database access, and data visualization. A connection was then established to the SQLite database containing the Maji Ndogo weather station and agricultural data.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text 
import matplotlib.pyplot as plt
import seaborn as sns

# Create a connection to the SQLite database
engine = create_engine('sqlite:///Maji_Ndogo_farm_survey_small.db')

## Testing the Database Connection

Before loading the data, I checked that the connection to the database was working correctly. This helped confirm that the data was ready for analysis.

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT name FROM sqlite_master WHERE type='table';"))
    for row in result:
        print(row)

## Loading the Data

To load the dataset, I used an SQL query that avoids duplicate `Field_ID` values. This helps ensure that each field appears only once in the dataset before moving on to the analysis.

In [ ]:
sql_query = """
SELECT *
FROM geographic_features
LEFT JOIN weather_features USING (Field_ID)
LEFT JOIN soil_and_crop_features USING (Field_ID)
LEFT JOIN farm_management_features USING (Field_ID)
"""

# Load the data into a Pandas DataFrame
with engine.connect() as connection:
    MD_agric_df = pd.read_sql_query(text(sql_query), connection)

## Cleaning the Data

Before analysing the dataset, I corrected a few data quality issues. This included fixing column names, correcting spelling mistakes in crop names, and ensuring numerical values were stored in the correct format.

In [ ]:
MD_agric_df.rename(columns={'Annual_yield': 'Crop_type_Temp', 'Crop_type': 'Annual_yield'}, inplace=True)
MD_agric_df.rename(columns={'Crop_type_Temp': 'Crop_type'}, inplace=True)
MD_agric_df['Elevation'] = MD_agric_df['Elevation'].abs()

def correct_crop_type(crop):
    crop = crop.strip()
    corrections = {
        'cassaval': 'cassava',
        'wheatn': 'wheat',
        'teaa': 'tea'
    }
    return corrections.get(crop, crop)

MD_agric_df['Crop_type'] = MD_agric_df['Crop_type'].apply(correct_crop_type)

In [ ]:
MD_agric_df

## Exploratory Data Analysis

After cleaning the data, I explored the dataset to better understand its structure. I started by using `describe()` to view the summary statistics of the numerical features and get an overall picture of the dataset.

In [ ]:
MD_agric_df.describe()

## Exploring the Data Distribution

After reviewing the summary statistics, I explored how the numerical variables were distributed. Visualising the data helped me identify patterns, spot potential outliers, and better understand the characteristics of each feature before continuing with the analysis.

To do this, I used Seaborn to create distribution plots for the numerical variables.


In [ ]:
# Create a grid of plots
fig, axes = plt.subplots(4, 4, figsize=(15, 15))

### Visualising the Data

To better understand the dataset, I plotted the distribution of each numerical feature using KDE plots. I also included the mean value in each plot to make it easier to compare the centre of the distributions.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(15, 15))
axes = axes.flatten()

numerical_columns = MD_agric_df.select_dtypes(include='number')

for i, column in enumerate(numerical_columns.columns):
    sns.kdeplot(data=numerical_columns, x=column, ax=axes[i])

    mean_val = MD_agric_df[column].mean()
    axes[i].axvline(mean_val, color='red', linestyle='dashed', linewidth=2)

    axes[i].set_title(column)

plt.tight_layout()
plt.show()

### Visualising Feature Distributions

I used KDE plots to visualise the distribution of the numerical features in the dataset. This helped me understand the spread and shape of each variable before performing further analysis.`

In [ ]:
numerical_columns = MD_agric_df.drop(columns=["Field_ID"])
numerical_columns = numerical_columns.select_dtypes('number')

fig, axes = plt.subplots(4, 4, figsize=(15, 15))
axes = axes.flatten()

for i, column in enumerate(numerical_columns.columns):
    sns.kdeplot(data=numerical_columns, x=column, ax=axes[i])
    axes[i].set_title(column)

plt.tight_layout()
plt.show()

## Distribution Insights

The KDE plots help us understand how the values of different features are spread across the dataset.

- **Slope:** The distribution is slightly left-skewed, meaning most values are higher, but a few lower values affect the average. Because of this, the median may be a better measure than the mean when analysing this feature.

- **Rainfall:** The distribution looks mostly normal but has multiple peaks. This suggests that rainfall patterns may differ across different groups in the dataset, so we will explore this further by comparing locations and crop types.

- **Other Features:** Looking at distributions helps us identify unusual patterns, skewed data, and variables that may need further investigation before modelling.

## Rainfall Distribution by Location and Crop Type

The rainfall distribution shows multiple peaks, so we will separate the data by location and crop type to understand where these differences come from.

In [ ]:
sns.kdeplot(
    data=MD_agric_df,
    x="Rainfall",
    hue="Crop_type"
)

plt.title("Rainfall Distribution by Crop Type")
plt.xlabel("Rainfall")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

## Notes from Rainfall Analysis

**Location:**  
When we split the rainfall data by location, we can see differences in rainfall patterns between regions. This helps us understand whether some areas receive more rainfall than others.

**Crop Type:**  
When we split the rainfall data by crop type, we can observe whether certain crops are associated with specific rainfall conditions.

## Comparing Average Rainfall by Location

The KDE plot gives us a visual comparison of rainfall patterns across locations. To better understand these differences, we will calculate the average rainfall for each location and compare the results.

In [ ]:
rainfall_by_location = MD_agric_df.groupby("Location")["Rainfall"].mean()

rainfall_by_location

## Notes from Rainfall Comparison

- **Rural_Amanzi:** This location has the lowest average rainfall (about 724 mm). Crops grown here may be better suited to areas with less rainfall.

- **Rural_Sokoto:** This location has the highest average rainfall (about 1,705 mm), followed by Rural_Akatsi. Higher rainfall may affect which crops grow well in these areas.

- **Rainfall Differences:** Rainfall levels are different across locations, which may help explain why crop patterns vary between areas.

## Interpreting KDE Plots

The height of a KDE plot shows the density of values, not the number of records. The main things to look at are the shape of the distribution and the peaks, which can help identify patterns.

Features with multiple peaks will be explored further to understand what may be causing these differences.

## Multivariate Analysis

So far, we have seen that rainfall is linked to both location and crop type. In this section, we will explore these relationships in more detail.

## Rainfall by Crop Type

This plot shows how rainfall differs across different crop types and helps us compare their rainfall patterns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))

sns.violinplot(
    data=MD_agric_df,
    x="Crop_type",
    y="Rainfall"
)

plt.title("Rainfall Distribution by Crop Type")
plt.xlabel("Crop Type")
plt.ylabel("Rainfall")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Notes

- **Rice:** Rice is mostly grown in areas with around 1,600 mm of annual rainfall, suggesting it may prefer higher rainfall levels.

- **Coffee:** Coffee is found across a wider range of rainfall levels, which may indicate that it can grow under different rainfall conditions.

- **Bananas:** The rainfall distribution for bananas can also be compared to see the conditions where they are most commonly grown.

Other options to visualise categorical/continuous data are scatter plots, FacetGrids, or Bubble charts. Be adventurous and try one of these!

## Continuous Relationships

Next, we will look at the relationships between the numerical features using a pair plot. The points are coloured by crop type to make it easier to compare different crops.

In [ ]:
remove_columns =['Field_ID','Latitude','Longitude','Annual_yield']

sns.pairplot(MD_agric_df.drop(columns =remove_columns), hue="Crop_type")

## Notes

- **Elevation and Temperature:** Elevation appears to have a clear relationship with minimum temperature. As elevation changes, the minimum temperature also changes.

- **Crop Types:** Some crop types form clear clusters. For example, tea is mostly grouped between a pH of 4 and 6, with standard yields between 0.6 and 0.8. This suggests that tea grows well under these conditions.

- **Other Relationships:** Some variables show patterns, but they are not easy to explain from the pair plot alone. These relationships may need further analysis.

## Categorical Relationships

We can also explore the relationships between categorical variables. A cross-tabulation helps us compare different categories and see how often they appear together.

In [ ]:
pd.crosstab(MD_agric_df['Location'],MD_agric_df['Crop_type'])

## Notes

- **Rural_Amanzi:** Potatoes are the most common crop, followed by wheat and maize. This matches the lower rainfall levels in this location.

- **Rural_Kilimani:** Potatoes are the most common crop, followed by wheat. Cassava and tea are also common in this location.

- **Rural_Sokoto:** Tea is the most common crop by a large margin. Coffee and bananas are also grown more often than most other crops.

- **Rural_Hawassa:** Wheat is the most common crop, followed by bananas and cassava.

- **Rural_Akatsi:** Bananas are the most common crop, followed by wheat and coffee.

## Factors Affecting Crop Yield

So far, we have explored the relationships between different variables. Now, we will focus on the factors that may affect crop yield.

To do this, we will use a correlation matrix to see how the numerical variables are related to each other and to standard yield.

In [ ]:
MD_agric_df.corr(numeric_only=True)

## Correlation with Standard Yield

To better understand which factors are most closely related to crop yield, we will look at the correlation between **Standard_yield** and the other numerical variables.

In [ ]:
standard_yield_corr = (
    MD_agric_df.corr(numeric_only=True)["Standard_yield"]
    .sort_values(ascending=False)
)

standard_yield_corr

## Notes

- **Pollution_level:** Higher pollution is linked to lower crop yields, although the relationship is weak.

- **Min_temperature_C:** Higher minimum temperatures are linked to slightly higher crop yields. This suggests that crops may perform better when temperatures do not get too low.

- **Overall:** No single feature explains crop yield on its own. Most relationships are weak, which suggests that crop yield is influenced by several factors rather than just one.

## Coffee Crop Analysis

The analysis so far has looked at all crops together. To better understand what affects crop yield, we will focus on one crop at a time.

In this section, we will analyse coffee and explore how different features are related to its yield.

In [ ]:
coffee_df = MD_agric_df.query("Crop_type == 'coffee'")
coffee_df = coffee_df.drop(columns = ['Crop_type','Field_ID','Annual_yield'])

In [ ]:
sns.pairplot(coffee_df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
sns.kdeplot(data=MD_agric_df, x='Rainfall', hue='Soil_type', fill=True)
plt.title('Rainfall Distribution by Soil Type')
plt.xlabel('Rainfall (mm)')
plt.ylabel('Density')
plt.show()

## Notes

- Coffee yields are generally higher in areas with more rainfall.

- Coffee also performs better in more fertile soil.

- Higher pollution levels are linked to lower coffee yields.

## Conclusion

This analysis explored how different environmental and farm management factors relate to crop yield. The results show that crop performance is influenced by several factors, including rainfall, soil fertility, temperature, and pollution.

The analysis also showed that different crops perform well under different conditions. Looking at each crop separately provides a better understanding of the factors that may influence its yield.

## Weather Data Validation

### Introduction

Before continuing the analysis, it is important to check the quality of the weather data. We will compare it with data collected from weather stations to make sure it is reliable.

In [ ]:
# weather_station_df = pd.read_csv("Weather_station_data.csv")
weather_station_df = pd.read_csv("https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv")

## Weather Station Messages

The weather station data includes the station ID and the raw sensor messages. The messages are shown below.

In [ ]:
pd.set_option('display.max_colwidth', None)

weather_station_df

## Understanding the Messages

The messages contain a timestamp, a description, and a measurement value. They include temperature, rainfall, and air quality readings, but they are written in different formats.

Before using the data, we need to extract the measurement values and identify the type of reading in each message.

Our main dataset already includes average temperature, rainfall, and pollution values. We will compare these with the weather station data to check that the information is consistent.

The table below shows which weather station is linked to each field in the main dataset.

In [ ]:
# weather_station_mapping_df = pd.read_csv("Weather_data_field_mapping.csv")
weather_station_mapping_df = pd.read_csv("https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv")

In [ ]:
weather_station_mapping_df

## Data Extraction

To validate the weather data, we first need to extract the measurement values from the weather station messages. We will then compare the results with the main dataset to check that the data is consistent.

In [ ]:
weather_station_df.head(50)

## Extracting Temperature Data

The weather station messages use different formats to record temperature values. A regular expression (regex) is used to identify these messages and extract the temperature values.

The extraction function is then applied to all messages to identify the measurement type and store the extracted values in separate columns for further analysis.

In [ ]:
patterns = {
        'Temperature': r'(\d+(\.\d+)?)\s?C'
    }

In [ ]:
import re
import numpy as np

weather_station_df = pd.read_csv("Weather_station_data.csv")

def extract_measurement(message):
    """
    Extract the measurement type and value from a weather station message.

    Returns:
        tuple: (measurement type, measurement value). If no match is found,
        returns (None, None).
    """
    for key, pattern in patterns.items():
        match = re.search(pattern, message)
        if match:
            return key, float(next((x for x in match.groups() if x is not None)))

    return None, None


extracted_measurements = weather_station_df["Message"].apply(extract_measurement)

weather_station_df["Measurement"] = extracted_measurements.apply(lambda x: x[0])
weather_station_df["Value"] = extracted_measurements.apply(lambda x: x[1])

In [ ]:
weather_station_df

## Results

The temperature measurements were extracted successfully and stored in the **Measurement** and **Value** columns. The next step is to create patterns for the remaining measurement types, including rainfall and air quality.

In [ ]:
patterns = {
    "Temperature": r"(\d+(\.\d+)?)\s?C",
    "Rainfall": r"(\d+(\.\d+)?)\s?mm",
    "Pollution_level": r"(?:Pollution|Air Quality Index|Pollution Index).*?= (\d+(\.\d+)?)"
}

patterns

In [ ]:
extracted_measurements = weather_station_df["Message"].apply(extract_measurement)

weather_station_df["Measurement"] = extracted_measurements.apply(lambda x: x[0])
weather_station_df["Value"] = extracted_measurements.apply(lambda x: x[1])

## Checking the Results

To confirm that the extraction worked correctly, we will check whether any messages were not matched by the regex patterns.

In [ ]:
# Use this line of code to see which messages are not assigned yet.
weather_station_df[(weather_station_df['Measurement'] == None)|(weather_station_df['Value'].isna())]

## Comparing Mean Measurements

### Weather Station Data Preparation

After extracting the measurement values, we can calculate the average temperature, rainfall, and pollution level for each weather station. These averages will be used to compare the weather station data with the main dataset.

In [ ]:
weather_station_means = weather_station_df.groupby(
    ['Weather_station_ID', 'Measurement']
)['Value'].mean()

weather_station_means = weather_station_means.unstack()

weather_station_means

## Main Dataset Preparation

To compare weather station measurements with farm-level data, the weather station ID needs to be added to the main agricultural dataset.

This connection allows each farm field to be linked to its corresponding weather station, making it possible to compare weather patterns across both datasets.

In [ ]:
print("MD_agric_df columns")
print(MD_agric_df.columns)
print("\n")
print("weather_station_mapping_df columns")
print(weather_station_mapping_df.columns)

### Adding Weather Station Information

The weather station mapping table was merged with the agricultural dataset using the `Field_ID` column.

This links each farm field to its corresponding weather station, allowing weather measurements to be compared between the two datasets.

In [ ]:
import pandas as pd

MD_agric_df = pd.merge(
    MD_agric_df,
    weather_station_mapping_df[['Field_ID', 'Weather_station']],
    on='Field_ID',
    how='left'
)

MD_agric_df = MD_agric_df.loc[:, ~MD_agric_df.columns.str.contains('^Unnamed')]
MD_agric_df = MD_agric_df.loc[:, ~MD_agric_df.columns.duplicated()]

print(MD_agric_df.shape)

The merged dataset was checked to confirm that the merge completed successfully and that unnecessary duplicate or unnamed columns had been removed.

## Calculating Mean Values by Weather Station

The agricultural dataset was grouped by weather station to calculate the average rainfall, temperature, and pollution level for each station. These averages will be compared with the weather station measurements to check for consistency between the two datasets.

In [ ]:
MD_agric_df['Weather_station'].value_counts()

In [ ]:
MD_agric_df_weather_means = MD_agric_df.groupby("Weather_station").mean(numeric_only = True)[['Pollution_level','Rainfall', 'Ave_temps']]

MD_agric_df_weather_means = MD_agric_df_weather_means.rename(columns = {'Ave_temps':"Temperature"})
MD_agric_df_weather_means

## Comparing the Datasets

The average weather measurements from the agricultural dataset were compared with the corresponding weather station averages.

This comparison was used to verify that the merged data was consistent with the original weather station records and to identify any significant differences between the two datasets.

In [ ]:
def within_tolerance_percentage(extracted, original, tolerance_pct):
    """
    Returns True if the percentage difference between two values is within
    the specified tolerance.
    """
    percent_diff = abs((extracted - original) / original) * 100
    return percent_diff <= tolerance_pct


def check_means(MD_agric_df_weather_means, weather_station_means):
    """
    Compares the average weather measurements from the agricultural dataset
    with the weather station dataset and reports whether each value is
    within the specified tolerance.
    """
    true_count = 0
    false_count = 0

    for index, row in weather_station_means.iterrows():
        print(f"Weather Station ID: {index}")

        for measurement in row.index:
            extracted_mean = row[measurement]
            original_mean = MD_agric_df_weather_means.loc[index, measurement]

            within_spec = within_tolerance_percentage(
                extracted_mean,
                original_mean,
                tolerance_pct
            )

            if within_spec:
                true_count += 1
            else:
                false_count += 1

            print(
                f"Measurement: {measurement}, "
                f"Extracted Mean: {extracted_mean}, "
                f"Original Mean: {original_mean}, "
                f"Within Spec: {within_spec}"
            )
            print()

    print(f"True: {true_count}, False: {false_count}")


# Set the acceptable percentage difference
tolerance_pct = 1.5

# Compare the datasets
check_means(MD_agric_df_weather_means, weather_station_means)

## Comparison Results

The comparison showed that some weather measurements matched closely, while others differed beyond the acceptable tolerance level.

These discrepancies suggest that further investigation is needed to identify possible differences in data collection, processing, or data quality. The findings highlight the importance of validating data from multiple sources before using it for analysis or decision-making.